In [1]:
%pip install bleak


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 850.3/850.3 kB 5.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [bleak]
Note: you may need to restart the kernel to use updated packages.


In [8]:
import asyncio
from bleak import BleakScanner

async def discover_devices():
    print("Scanning for BLE devices... (this takes 5 seconds)")
    devices = await BleakScanner.discover()
    for d in devices:
        if d.name and "VEEPEAK" in d.name.upper():
            print(f"Found Adapter! Name: {d.name} | BLE Address: {d.address}")

# Run the async function inside Jupyter
await discover_devices()

Scanning for BLE devices... (this takes 5 seconds)
Found Adapter! Name: VEEPEAK | BLE Address: 66:1E:87:06:1A:05


In [6]:
import asyncio
from bleak import BleakClient

# Configuration constants for Veepeak BLE+
DEVICE_ADDRESS = "66:1E:87:06:1A:05"
# Standard Nordic UART UUIDs used by Veepeak to transmit/receive data
TX_UUID = "6e400003-b5a3-f393-e0a9-e50e24dcca9e" # Read from adapter
RX_UUID = "6e400002-b5a3-f393-e0a9-e50e24dcca9e" # Write to adapter

# Callback function to handle data coming back from the car
def notification_handler(sender, data):
    print(f"Car Response: {data.decode('utf-8', errors='ignore').strip()}")

async def connect_and_query_vehicle():
    print(f"Connecting to Veepeak at {DEVICE_ADDRESS}...")
    
    async with BleakClient(DEVICE_ADDRESS) as client:
        if client.is_connected:
            print("Connected to Veepeak over BLE successfully!")
            
            # Start listening for the car's data replies
            await client.start_notify(TX_UUID, notification_handler)
            
            # Send 'ATZ' command to test ELM327 initialization (requires car ignition ON)
            print("Sending reset command (ATZ) to ECU...")
            # ELM327 commands must end with a carriage return (\r)
            await client.write_gatt_char(RX_UUID, b"ATZ\r")
            
            # Wait 2 seconds to allow the car to respond to the console
            await asyncio.sleep(2)
            
            # Clean up the notification channel
            await client.stop_notify(TX_UUID)
        else:
            print("Could not establish a BLE handshake.")

# Execute the connection
await connect_and_query_vehicle()


Connecting to Veepeak at 66:1E:87:06:1A:05...
Connected to Veepeak over BLE successfully!


BleakCharacteristicNotFoundError: Characteristic 6e400003-b5a3-f393-e0a9-e50e24dcca9e was not found!

In [7]:
import asyncio
from bleak import BleakClient

# Use your Veepeak's BLE address found previously
DEVICE_ADDRESS = "66:1E:87:06:1A:05"

async def explore_services():
    print(f"Connecting to {DEVICE_ADDRESS} to read internal map...")
    async with BleakClient(DEVICE_ADDRESS) as client:
        if client.is_connected:
            print("Connected! Scanning services...\n")
            for service in client.services:
                print(f"[Service] {service.uuid}")
                for char in service.characteristics:
                    print(f"  └── [Characteristic] {char.uuid} | Properties: {char.properties}")
        else:
            print("Failed to connect.")

await explore_services()


Connecting to 66:1E:87:06:1A:05 to read internal map...


BleakDBusError: [org.bluez.Error.NotAvailable] br-connection-profile-unavailable

In [9]:
import asyncio
from bleak import BleakClient

# Use your Veepeak's BLE address found previously
DEVICE_ADDRESS = "66:1E:87:06:1A:05"

async def explore_services():
    print(f"Connecting to {DEVICE_ADDRESS} to read internal map...")
    async with BleakClient(DEVICE_ADDRESS) as client:
        if client.is_connected:
            print("Connected! Scanning services...\n")
            for service in client.services:
                print(f"[Service] {service.uuid}")
                for char in service.characteristics:
                    print(f"  └── [Characteristic] {char.uuid} | Properties: {char.properties}")
        else:
            print("Failed to connect.")

await explore_services()


Connecting to 66:1E:87:06:1A:05 to read internal map...
Connected! Scanning services...

[Service] 00001801-0000-1000-8000-00805f9b34fb
  └── [Characteristic] 00002a05-0000-1000-8000-00805f9b34fb | Properties: ['indicate']
[Service] 00006287-3c17-d293-8e48-14fe2e4da212
  └── [Characteristic] 00006387-3c17-d293-8e48-14fe2e4da212 | Properties: ['write-without-response']
  └── [Characteristic] 00006487-3c17-d293-8e48-14fe2e4da212 | Properties: ['write', 'notify']
[Service] 0000180a-0000-1000-8000-00805f9b34fb
  └── [Characteristic] 00002a50-0000-1000-8000-00805f9b34fb | Properties: ['read']
  └── [Characteristic] 00002a25-0000-1000-8000-00805f9b34fb | Properties: ['read']
  └── [Characteristic] 00002a23-0000-1000-8000-00805f9b34fb | Properties: ['read']
  └── [Characteristic] 00002a27-0000-1000-8000-00805f9b34fb | Properties: ['read']
  └── [Characteristic] 00002a29-0000-1000-8000-00805f9b34fb | Properties: ['read']
  └── [Characteristic] 00002a2a-0000-1000-8000-00805f9b34fb | Properties:

In [1]:
import asyncio
from bleak import BleakClient

# Configuration constants for Veepeak BLE+
DEVICE_ADDRESS = "66:1E:87:06:1A:05"
# Standard Nordic UART UUIDs used by Veepeak to transmit/receive data
TX_UUID = "0000fff1-0000-1000-8000-00805f9b34fb" # Notify
RX_UUID = "0000fff2-0000-1000-8000-00805f9b34fb" # Write

# Callback function to handle data coming back from the car
def notification_handler(sender, data):
    print(f"Car Response: {data.decode('utf-8', errors='ignore').strip()}")

async def connect_and_query_vehicle():
    print(f"Connecting to Veepeak at {DEVICE_ADDRESS}...")
    
    async with BleakClient(DEVICE_ADDRESS) as client:
        if client.is_connected:
            print("Connected to Veepeak over BLE successfully!")
            
            # Start listening for the car's data replies
            await client.start_notify(TX_UUID, notification_handler)
            
            # Send 'ATZ' command to test ELM327 initialization (requires car ignition ON)
            print("Sending reset command (ATZ) to ECU...")
            # ELM327 commands must end with a carriage return (\r)
            await client.write_gatt_char(RX_UUID, b"ATZ\r")
            
            # Wait 2 seconds to allow the car to respond to the console
            await asyncio.sleep(2)
            
            # Clean up the notification channel
            await client.stop_notify(TX_UUID)
        else:
            print("Could not establish a BLE handshake.")

# Execute the connection
await connect_and_query_vehicle()


Connecting to Veepeak at 66:1E:87:06:1A:05...
Connected to Veepeak over BLE successfully!


BleakError: Multiple Characteristics with this UUID, refer to your desired characteristic by the `handle` attribute instead.

In [5]:
import asyncio
from bleak import BleakClient

DEVICE_ADDRESS = "66:1E:87:06:1A:05"

async def find_handles():
    async with BleakClient(DEVICE_ADDRESS) as client:
        print("Mapping hardware handles...\n")
        for service in client.services:
            # We only care about the fff0 service group we found earlier
            if "fff0" in service.uuid:
                for char in service.characteristics:
                    print(f"UUID: {char.uuid} | HANDLE: {char.handle} | Properties: {char.properties}")

await find_handles()


Mapping hardware handles...

UUID: 0000fff1-0000-1000-8000-00805f9b34fb | HANDLE: 17 | Properties: ['notify']
UUID: 00002902-0000-1000-8000-00805f9b34fb | HANDLE: 20 | Properties: ['notify']
UUID: 0000fff2-0000-1000-8000-00805f9b34fb | HANDLE: 15 | Properties: ['write-without-response', 'write']


In [2]:
import getpass
import subprocess

password = getpass.getpass()

# Define your command as a list
cmd = ["sudo", "-S", "bash", "./bluetooth_reset.sh"]

# Run the command and feed the password directly into stdin as bytes
process = subprocess.Popen(
    cmd, stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True
)
stdout, stderr = process.communicate(input=password + "\n")

# Print the script's output
print(stdout)
if stderr:
    print("Errors:", stderr)


 ········


--- Starting Bluetooth Reset Script ---
Waiting to connect to bluetoothd...[bluetooth]# remove 66:1E:87:06:1A:05
Device 66:1E:87:06:1A:05 not available
[bluetooth]# 

[bluetooth]# exit
[bluetooth]# Restarting system Bluetooth service...
--- Bluetooth Successfully Reset! ---



In [15]:
import asyncio
from bleak import BleakClient

DEVICE_ADDRESS = "66:1E:87:06:1A:05"

TX_UUID = 20 # Notify
RX_UUID = 15 # Write

# Callback function to handle data coming back from the car
def notification_handler(sender, data):
    print(f"Car Response: {data.decode('utf-8', errors='ignore').strip()}")

async def connect_and_query_vehicle():
    print(f"Connecting to Veepeak at {DEVICE_ADDRESS}...")
    
    async with BleakClient(DEVICE_ADDRESS) as client:
        if client.is_connected:
            print("Connected to Veepeak over BLE successfully!")
            
            # Start listening for the car's data replies
            await client.start_notify(TX_UUID, notification_handler)
            
            # Send 'ATZ' command to test ELM327 initialization (requires car ignition ON)
            print("Sending reset command (ATZ) to ECU...")
            # ELM327 commands must end with a carriage return (\r)
            await client.write_gatt_char(RX_UUID, b"ATZ\r")
            
            # Wait 2 seconds to allow the car to respond to the console
            await asyncio.sleep(2)
            
            # Clean up the notification channel
            await client.stop_notify(TX_UUID)
        else:
            print("Could not establish a BLE handshake.")

# Execute the connection
await connect_and_query_vehicle()


Connecting to Veepeak at 66:1E:87:06:1A:05...
Connected to Veepeak over BLE successfully!
Sending reset command (ATZ) to ECU...


In [ ]:
import asyncio
from bleak import BleakClient

# Paste your correct, verified handles here
WRITE_HANDLE = 15   # Replace with your verified Write handle
NOTIFY_HANDLE = 17  # Replace with your verified Notify/Read handle

# This function will trigger automatically whenever the Veepeak responds
def notification_handler(sender, data):
    # Decode the bytes from the vehicle into readable text
    response_text = data.decode('utf-8', errors='ignore')
    print(f"<- Received from Veepeak: {repr(response_text)}")

async def run_obd2():
    device_address = "66:1E:87:06:1A:05"
    
    print(f"Connecting to Veepeak at {device_address}...")
    async with BleakClient(device_address) as client:
        print("Connected to Veepeak over BLE successfully!")
        
        # 1. Start listening on the notification handle BEFORE writing
        await client.start_notify(NOTIFY_HANDLE, notification_handler)
        await asyncio.sleep(0.5) # Give the BLE stack a brief moment to settle
        
        print("Sending reset command (ATZ) to ECU...")
        
        # 2. Append '\r' so the ELM327 chip knows the command is complete
        reset_command = b"ATZ\r" 
        await client.write_gatt_char(WRITE_HANDLE, reset_command, response=False)
        await client.write_gatt_char(WRITE_HANDLE, b"0100\r", response=False)
        
        # 3. Keep the script alive to listen for the incoming response
        # (Otherwise the program finishes and closes immediately)
        print("Waiting for response... (Press Ctrl+C to exit)")
        while True:
            await asyncio.sleep(1)

await run_obd2()

Connecting to Veepeak at 66:1E:87:06:1A:05...
Connected to Veepeak over BLE successfully!
Sending reset command (ATZ) to ECU...
Waiting for response... (Press Ctrl+C to exit)
<- Received from Veepeak: 'ATZ\r'
<- Received from Veepeak: '\r\rELM327 v2.2\r\r>'


In [12]:
obd_library = {
    "0105": {
        "name": "coolant_temp",
        "formula": lambda b: b[0] - 40,
        "unit": "°C"
    },
    "010C": {
        "name": "engine_rpm",
        "formula": lambda b: ((b[0] * 256) + b[1]) / 4,
        "unit": "RPM"
    }
}

# Executing a dynamic lookup
# pid = "010C"
# incoming_data = [26, 64] # Represents 1,700 RPM
# result = obd_library[pid]["formula"](incoming_data)
# unit = obd_library[pid]["unit"]

# print(f"{obd_library[pid]['name']}: {result} {unit}")

In [ ]:
import asyncio
import nest_asyncio
from bleak import BleakClient

nest_asyncio.apply()

WRITE_HANDLE = 15   
NOTIFY_HANDLE = 17  

data_queue = asyncio.Queue()

def notification_handler(sender, data):
    response_text = data.decode('utf-8', errors='ignore')

    asyncio.get_running_loop().call_soon_threadsafe(data_queue.put_nowait, response_text)

    response = b""
    while b">" not in response:
        if response.in_waiting > 0:
            response += ser.read(ser.in_waiting)
        time.sleep(0.02) # Small nap to prevent CPU spiking

async def send_command_and_wait(client, command_string, timeout=2.0):
    """Formats the user string, sends it, and waits for a response."""
    # Ensure command ends with a carriage return and convert to bytes
    if not command_string.endswith('\r'):
        command_string += '\r'
    command_bytes = command_string.encode('utf-8')
    
    await client.write_gatt_char(WRITE_HANDLE, command_bytes, response=False)
    try:
        return await asyncio.wait_for(data_queue.get(), timeout=timeout)
    except asyncio.TimeoutError:
        return "Error: No response from ECU (Timeout)"

async def get_user_input():
    """Runs the blocking input() function in a separate thread to keep async alive."""
    # This prevents the prompt from freezing your Bluetooth connection
    return await asyncio.to_thread(input, "\nEnter OBD2 command (e.g., '010C', '0100') or 'exit': ")

async def run_obd2():
    device_address = "66:1E:87:06:1A:05"
    
    async with BleakClient(device_address) as client:
        await client.start_notify(NOTIFY_HANDLE, notification_handler)
        await asyncio.sleep(1)
        
        # ==========================================
        # PHASE 1: INITIALIZATION (Happens ONCE)
        # ==========================================
        print("Initializing ELM327 chip...")
        atz_response = await send_command_and_wait(client, "ATZ", timeout=5.0) 
        print(f"Reset Response: {atz_response.strip()}")

        # Turn off echo to clean output buffer
        ate0_response = await send_command_and_wait(client,"ATE0", timeout=5.0)
        print(f"Reset Response: {ate0_response.strip()}")

        # Eliminate spaces for clean output
        ats0_response = await send_command_and_wait(client,"ATS0", timeout=5.0)
        print(f"Reset Response: {ats0_response.strip()}")
        # ==========================================
        # PHASE 2: INTERACTIVE USER PROMPT LOOP
        # ==========================================
        print("\n--- OBD2 Terminal Ready ---")
        print("Type standard OBD2 PIDs or AT commands.")
        
        while True:
            # Safely prompt the user
            user_command = await get_user_input()
            
            # Clean up input string
            user_command = user_command.strip()
            
            # Check for exit condition
            if user_command.lower() in ['exit', 'quit']:
                print("Exiting interactive terminal...")
                break
                
            if not user_command:
                continue
                
            print(f"Sending: {user_command}")
            response = await send_command_and_wait(client, user_command)
            
            if user_command in obd_library:
                # print what the dictionary has
                continue

            else:
                print(repr(response))
                clean_response = response.strip()
                print(f"Response: {clean_response}")
            
        # Cleanup
        await client.stop_notify(NOTIFY_HANDLE)
        print("Disconnected cleanly.")

try:
    asyncio.run(run_obd2())
except KeyboardInterrupt:
    print("\nProgram terminated.")


A message handler raised an exception: name 'ser' is not defined
Traceback (most recent call last):
  File "src/dbus_fast/message_bus.py", line 794, in dbus_fast.message_bus.BaseMessageBus._process_message
  File "/home/abner/anaconda3/lib/python3.13/site-packages/bleak/backends/bluezdbus/manager.py", line 1170, in _parse_msg
    watcher.on_characteristic_value_changed(
    ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        message_path, new_value
        ^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/home/abner/anaconda3/lib/python3.13/site-packages/bleak/backends/bluezdbus/client.py", line 191, in on_value_changed
    callback(bytearray(value))
    ~~~~~~~~^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_170428/3018975928.py", line 19, in notification_handler
    if ser.in_waiting > 0:
       ^^^
NameError: name 'ser' is not defined. Did you mean: 'set'?
A message handler raised an exception: name 'ser' is not defined
Traceback (most recent call last):
  File "src/dbus_fast/message_bus.py", lin

Initializing ELM327 chip...
Reset Response: ATZ
>eset Response: ELM327 v2.2
>Kset Response: ATE0

--- OBD2 Terminal Ready ---
Type standard OBD2 PIDs or AT commands.


In [ ]:
# note print the codes on the output so i am not scrolling up so much
# maybe I can use the dictionary to print out which ones are available
# So first do a 0100 

In [ ]:
import sys
# import obd

def connect_to_car():
    # Attempt connection to OBD port (e.g., /dev/ttyUSB0)
    connection = None  # placeholder for logic
    if not connection:
        print("Hardware Error: OBD-II scanner not detected.")
        sys.exit(10)  # Custom exit code for "No Hardware"

# use sys.exits to not get stuck with a stupid fat bill 

In [ ]:
python3 read_obd.py && python3 compress_data.py && python3 upload_via_lte.py
# how I want to split the files up, leverage linux kernel exit codes


In [ ]:
import time
import sys

def upload_packet(file_path):
    attempt = 0
    max_attempts = 5
    wait_time = 5  # Start with a 5-second delay

    while attempt < max_attempts:
        print(f"Attempting LTE upload (Attempt {attempt + 1})...")
        success = False # Placeholder for your actual upload function
        
        if success:
            print("Upload successful!")
            return True
            
        print(f"Upload failed. Network might be down. Waiting {wait_time}s...")
        time.sleep(wait_time)
        
        attempt += 1
        wait_time *= 2  # Double the wait time every failure (5s -> 10s -> 20s...)

    # If it fails completely, exit with a custom error code so Bash knows
    sys.exit(50) 
# This is so we're not spamming the pipeline if we lose signal 

In [ ]:
obd_library = {
    # --- ENGINE SYNCHRONIZATION & LOAD ---
    "0104": {
        "name": "calculated_engine_load",
        "formula": lambda b: (100 / 255) * b[0],
        "unit": "%"
    },
    "010C": {
        "name": "engine_rpm",
        "formula": lambda b: ((b[0] * 256) + b[1]) / 4,
        "unit": "RPM"
    },
    
    # --- DRIVER INPUTS & PERFORMANCE ---
    "010D": {
        "name": "vehicle_speed",
        "formula": lambda b: b[0],
        "unit": "km/h"
    },
    "0111": {
        "name": "throttle_position",
        "formula": lambda b: (100 / 255) * b[0],
        "unit": "%"
    },
    
    # --- THERMAL CRITICAL BOUNDARIES (Poll these slowly) ---
    "0105": {
        "name": "coolant_temp",
        "formula": lambda b: b[0] - 40,
        "unit": "°C"
    },
    "010F": {
        "name": "intake_air_temperature",
        "formula": lambda b: b[0] - 40,
        "unit": "°C"
    }
}

In [3]:
import asyncio
import nest_asyncio
from bleak import BleakClient

nest_asyncio.apply()

WRITE_HANDLE = 15   
NOTIFY_HANDLE = 17

data_queue = asyncio.Queue()

# 1. Keep the handler non-blocking. It just catches packets and forwards them.
def notification_handler(sender, data):
    response_text = data.decode('utf-8', errors='ignore')
    # Safely push the incoming string chunk into our async queue
    asyncio.get_running_loop().call_soon_threadsafe(data_queue.put_nowait, response_text)

# 2. Accumulate chunks here until the ELM327 prompt (">") is found
async def send_command_and_wait(client, command_string, timeout=2.0):
    """Formats the user string, sends it, and waits for a response ending in '>'."""
    if not command_string.endswith('\r'):
        command_string += '\r'
    command_bytes = command_string.encode('utf-8')
    
    # Flush any leftover junk from the queue before sending a new command
    while not data_queue.empty():
        data_queue.get_nowait()
    
    await client.write_gatt_char(WRITE_HANDLE, command_bytes, response=False)
    
    full_response = ""
    try:
        # Keep reading chunks from the queue until we see the ELM327 prompt
        while ">" not in full_response:
            # Wait for the next chunk of data from Bleak
            chunk = await asyncio.wait_for(data_queue.get(), timeout=timeout)
            full_response += chunk
            
        return full_response
    except asyncio.TimeoutError:
        # Return whatever we managed to grab before timing out, or an error
        if full_response:
            return full_response + "\n[Error: Timeout waiting for total response]"
        return "Error: No response from ECU (Timeout)"

async def get_user_input():
    """Runs the blocking input() function in a separate thread to keep async alive."""
    return await asyncio.to_thread(input, "\nEnter OBD2 command (e.g., '010C', '0100') or 'exit': ")

async def run_obd2():
    device_address = "66:1E:87:06:1A:05"
    
    async with BleakClient(device_address) as client:
        await client.start_notify(NOTIFY_HANDLE, notification_handler)
        await asyncio.sleep(1)
        
        # ==========================================
        # PHASE 1: INITIALIZATION (Happens ONCE)
        # ==========================================
        print("Initializing ELM327 chip...")
        atz_response = await send_command_and_wait(client, "ATZ", timeout=5.0) 
        print(f"Reset Response:\n{atz_response.strip()}")

        ate0_response = await send_command_and_wait(client, "ATE0", timeout=5.0)
        print(f"Echo Off Response:\n{ate0_response.strip()}")

        ats0_response = await send_command_and_wait(client, "ATS0", timeout=5.0)
        print(f"Spaces Off Response:\n{ats0_response.strip()}")
        
        # ==========================================
        # PHASE 2: INTERACTIVE USER PROMPT LOOP
        # ==========================================
        print("\n--- OBD2 Terminal Ready ---")
        print("Type standard OBD2 PIDs or AT commands.")
        
        while True:
            user_command = await get_user_input()
            user_command = user_command.strip()
            
            if user_command.lower() in ['exit', 'quit']:
                print("Exiting interactive terminal...")
                break
                
            if not user_command:
                continue
                
            print(f"Sending: {user_command}")
            response = await send_command_and_wait(client, user_command)
            

            print(f"Raw: {repr(response)}")
            clean_response = response.replace(">", "").strip()
            print(f"Response: {clean_response}")
            
        await client.stop_notify(NOTIFY_HANDLE)
        print("Disconnected cleanly.")

try:
    asyncio.run(run_obd2())
except KeyboardInterrupt:
    print("\nProgram terminated.")


Initializing ELM327 chip...
Reset Response:
>LM327 v2.2
Echo Off Response:
>KE0
Spaces Off Response:
>K

--- OBD2 Terminal Ready ---
Type standard OBD2 PIDs or AT commands.



Enter OBD2 command (e.g., '010C', '0100') or 'exit':  0100


Sending: 0100
Raw: '4100BF9FA893\r\r>'
Response: 4100BF9FA893



Enter OBD2 command (e.g., '010C', '0100') or 'exit':  0104


Sending: 0104
Raw: '41045D\r\r>'
Response: 41045D


...

Sending: 0105
Raw: '41055D\r\r>'
Response: 41055D



Enter OBD2 command (e.g., '010C', '0100') or 'exit':  010F


Sending: 010F
Raw: '410F49\r\r>'
Response: 410F49



Enter OBD2 command (e.g., '010C', '0100') or 'exit':  exit


Exiting interactive terminal...
Disconnected cleanly.


In [10]:
import struct
import time

def pack_raw_obd_string(raw_obd_line):
    """
    Packs a vehicle metric event into a fixed 8-byte binary layout.
    
    Layout Format Specifier: '>IHH'
    - '>' : Big-endian byte format (Network Standard)
    - 'I' : 4-byte Unsigned Integer (Unix Timestamp)
    - 'H' : 2-byte Unsigned Short (PID Identifier, e.g., 0x010C)
    - 'H' : 2-byte Unsigned Short (Raw Value Data Payload)
    
    Total Packet Size = 4 + 2 + 2 = 8 Bytes.
    """
    timestamp = int(time.time())
    
    # 1. Extract the PID from characters 2 and 3 (index 2:4)
    pid_hex = raw_obd_line[2:4]
    pid_int = int(pid_hex, 16)
    
    # 2. Extract the data payload (everything after the PID)
    data_hex = raw_obd_line[4:]
    data_int = int(data_hex, 16) # Automatically handles both 1-byte and 2-byte hex widths
    
    # 3. Pack into fixed-width network format: >IHH (4-byte time, 2-byte PID, 2-byte data)
    packed_binary = struct.pack('>IHH', timestamp, pid_int, data_int)
    
    return packed_binary



# --- SIMULATION EXAMPLE (Onboard the Car) ---
# Testing with an Engine RPM reading ("010C") returning a value payload of "0B64" (2916 RPM)
mock_pid = "010C"
mock_hex_val = "0DFA"

binary_packet = pack_telemetry_packet(mock_pid, mock_hex_val)

print("--- ONBOARD TRANSMISSION OUTBOUND ---")
print(f"Original Text: {mock_pid},{mock_hex_val} (~10-25 bytes depending on layout formatting)")
print(f"Packed Packet: {binary_packet}")
print(f"Payload Size : {len(binary_packet)} bytes flat.")


# --- RECEIVING EXAMPLE (What the Cloud Guy Runs) ---
# The server receives the 8 bytes and extracts the exact numbers instantly
unpacked_tuple = struct.unpack('>IHH', binary_packet)

print("\n--- CLOUD INGESTION INBOUND ---")
print(f"Unpacked Data Tuple: {unpacked_tuple}")
print(f"Extracted Time : {unpacked_tuple[0]}")
print(f"Extracted PID  : 0{unpacked_tuple[1]:02X}")  # Converts back to standard string format
print(f"Extracted Hex  : {unpacked_tuple[2]:04X}")

--- ONBOARD TRANSMISSION OUTBOUND ---
Original Text: 010C,0DFA (~10-25 bytes depending on layout formatting)
Packed Packet: b'j\xafB\xab\x01\x0c\r\xfa'
Payload Size : 8 bytes flat.

--- CLOUD INGESTION INBOUND ---
Unpacked Data Tuple: (1789870763, 268, 3578)
Extracted Time : 1789870763
Extracted PID  : 010C
Extracted Hex  : 0DFA
